# Module 13.3: Model Quantization Comparison

**Duration**: 1 hour  
**Focus**: Confronto di modelli con quantizzazione diversa (ottimizzato per CPU)

---

## Obiettivi di Apprendimento

- Comprendere cos'è la quantizzazione dei modelli
- Confrontare modelli con precision diversa (FP32, FP16, INT8)
- Valutare il trade-off tra velocità, memoria e qualità
- Misurare performance quantitative su task di generazione

---

## Cos'è la Quantizzazione?

**Quantizzazione** è una tecnica per ridurre la precisione numerica dei parametri del modello:

- **FP32** (Float32): Precisione standard, 32 bit per parametro
- **FP16** (Float16): Metà precisione, 16 bit per parametro
- **INT8**: Interi 8-bit, massive compression
- **INT4**: Estrema compressione, quality loss significativo

**Vantaggi**: Meno memoria, inferenza più veloce, deployment su dispositivi limitati  
**Svantaggi**: Potenziale degradazione della qualità

In [ ]:
# Installazione dipendenze (esegui solo la prima volta)
!pip install transformers torch bitsandbytes accelerate
!pip install matplotlib seaborn pandas numpy psutil
!pip install sacrebleu rouge-score

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import psutil
import os
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from sacrebleu import sentence_bleu
from rouge_score import rouge_scorer
import warnings
warnings.filterwarnings('ignore')

# Configurazione per riproducibilità
torch.manual_seed(42)
np.random.seed(42)

# Check hardware
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CPU cores: {psutil.cpu_count()}")
print(f"RAM available: {psutil.virtual_memory().available / (1024**3):.1f} GB")

## 1. Preparazione Dataset di Test

Creiamo un set di prompt diversificati per testare la qualità di generazione.

In [ ]:
# Dataset di test con diverse tipologie di task
test_prompts = [
    {
        "category": "Creative Writing",
        "prompt": "Write a short story about a robot learning to paint:",
        "expected_style": "narrative, descriptive"
    },
    {
        "category": "Technical Explanation", 
        "prompt": "Explain how neural networks work in simple terms:",
        "expected_style": "clear, educational"
    },
    {
        "category": "Code Generation",
        "prompt": "Write a Python function to calculate fibonacci numbers:",
        "expected_style": "precise, syntactic"
    },
    {
        "category": "Reasoning",
        "prompt": "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?",
        "expected_style": "logical, step-by-step"
    },
    {
        "category": "Summary",
        "prompt": "Summarize the main benefits of renewable energy:",
        "expected_style": "concise, informative"
    },
    {
        "category": "Translation",
        "prompt": "Translate to Italian: The weather is beautiful today and perfect for a walk in the park.",
        "expected_style": "accurate, natural"
    }
]

print(f"Prepared {len(test_prompts)} test prompts covering different domains:")
for prompt in test_prompts:
    print(f"- {prompt['category']}: {prompt['expected_style']}")

## 2. Utility Functions per Benchmarking

In [ ]:
def get_model_size_mb(model):
    """Calcola dimensione del modello in MB"""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    return (param_size + buffer_size) / (1024**2)

def measure_memory_usage():
    """Misura uso memoria corrente"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024**2)  # MB

def generate_with_timing(model, tokenizer, prompt, max_length=100, **kwargs):
    """Genera testo misurando tempo e memoria"""
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Memoria prima
    mem_before = measure_memory_usage()
    
    # Generazione con timing
    start_time = time.time()
    
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            max_length=max_length,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )
    
    end_time = time.time()
    
    # Memoria dopo
    mem_after = measure_memory_usage()
    
    # Decodifica
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response_only = generated_text.replace(prompt, "").strip()
    
    return {
        'text': response_only,
        'generation_time': end_time - start_time,
        'memory_used': mem_after - mem_before,
        'tokens_generated': len(outputs[0]) - len(inputs.input_ids[0])
    }

def calculate_text_metrics(text):
    """Calcola metriche di base del testo"""
    words = text.split()
    sentences = text.split('.') 
    
    return {
        'word_count': len(words),
        'sentence_count': len([s for s in sentences if s.strip()]),
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
        'vocabulary_richness': len(set(words)) / len(words) if words else 0
    }

print("Utility functions defined successfully!")

## 3. Caricamento Modelli con Diverse Quantizzazioni

Caricheremo lo stesso modello (DistilGPT-2) con diverse precisioni per confrontare.

In [ ]:
# Scegliamo DistilGPT-2 per essere gentili con la CPU
model_name = "distilgpt2"  # ~82M parametri, più leggero di GPT-2

# Carica tokenizer (uguale per tutti)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

models = {}

print("Loading models with different quantizations...")

# 1. Modello FP32 (full precision)
print("\n1. Loading FP32 model...")
models['FP32'] = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="cpu"
)
print(f"   Size: {get_model_size_mb(models['FP32']):.1f} MB")

# 2. Modello FP16 (half precision) - se supportato
print("\n2. Loading FP16 model...")
try:
    models['FP16'] = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="cpu"
    )
    print(f"   Size: {get_model_size_mb(models['FP16']):.1f} MB")
except Exception as e:
    print(f"   FP16 not supported on CPU, using FP32 instead")
    models['FP16'] = models['FP32']  # Fallback

# 3. Modello INT8 (8-bit quantization)
print("\n3. Loading INT8 model...")
try:
    quantization_config = BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=True
    )
    models['INT8'] = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quantization_config,
        device_map="auto"
    )
    print(f"   Size: {get_model_size_mb(models['INT8']):.1f} MB (estimated)")
except Exception as e:
    print(f"   INT8 quantization failed: {e}")
    print(f"   Using FP32 model as fallback")
    models['INT8'] = models['FP32']  # Fallback

print(f"\nSuccessfully loaded {len(models)} model variants:")
for name, model in models.items():
    print(f"- {name}: {get_model_size_mb(model):.1f} MB")

## 4. Benchmarking Performance

Testiamo ogni modello su tutti i prompt e misuriamo performance.

In [ ]:
# Eseguiamo il benchmark completo
results = []

print("Starting comprehensive benchmark...")
print("This may take several minutes on CPU.\n")

for model_name, model in models.items():
    print(f"Testing {model_name} model...")
    
    for i, test_case in enumerate(test_prompts):
        print(f"  - {test_case['category']} ({i+1}/{len(test_prompts)})")
        
        # Genera con timing
        generation_result = generate_with_timing(
            model, tokenizer, 
            test_case['prompt'],
            max_length=150
        )
        
        # Calcola metriche del testo
        text_metrics = calculate_text_metrics(generation_result['text'])
        
        # Combina tutti i risultati
        result = {
            'model': model_name,
            'category': test_case['category'],
            'prompt': test_case['prompt'][:50] + '...',
            'generated_text': generation_result['text'][:100] + '...',
            'generation_time': generation_result['generation_time'],
            'memory_used': generation_result['memory_used'],
            'tokens_generated': generation_result['tokens_generated'],
            'tokens_per_second': generation_result['tokens_generated'] / generation_result['generation_time'],
            **text_metrics
        }
        
        results.append(result)
        
        # Pausa breve per stabilità
        time.sleep(0.5)

print("\nBenchmark completed!")
print(f"Total results: {len(results)}")

# Converti in DataFrame per analisi
df = pd.DataFrame(results)
print("\nFirst few results:")
print(df[['model', 'category', 'generation_time', 'tokens_per_second', 'word_count']].head())

## 5. Analisi Performance: Velocità e Efficienza

In [ ]:
# Calcola statistiche per modello
model_stats = df.groupby('model').agg({
    'generation_time': ['mean', 'std'],
    'tokens_per_second': ['mean', 'std'],
    'memory_used': ['mean', 'std'],
    'word_count': ['mean', 'std'],
    'vocabulary_richness': ['mean', 'std']
}).round(3)

model_stats.columns = ['_'.join(col).strip() for col in model_stats.columns]

print("=== PERFORMANCE STATISTICS BY MODEL ===")
print(model_stats)

# Visualizzazioni
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Tempo di generazione
sns.boxplot(data=df, x='model', y='generation_time', ax=axes[0,0])
axes[0,0].set_title('Generation Time by Model')
axes[0,0].set_ylabel('Time (seconds)')

# 2. Token per secondo
sns.boxplot(data=df, x='model', y='tokens_per_second', ax=axes[0,1])
axes[0,1].set_title('Throughput (Tokens/Second)')
axes[0,1].set_ylabel('Tokens per second')

# 3. Memoria usata
sns.boxplot(data=df, x='model', y='memory_used', ax=axes[1,0])
axes[1,0].set_title('Memory Usage During Generation')
axes[1,0].set_ylabel('Memory (MB)')

# 4. Lunghezza output
sns.boxplot(data=df, x='model', y='word_count', ax=axes[1,1])
axes[1,1].set_title('Output Length (Words)')
axes[1,1].set_ylabel('Word count')

plt.tight_layout()
plt.show()

# Confronto efficienza
efficiency_comparison = model_stats[['generation_time_mean', 'tokens_per_second_mean', 'memory_used_mean']].copy()
efficiency_comparison['efficiency_score'] = (
    efficiency_comparison['tokens_per_second_mean'] / 
    (efficiency_comparison['generation_time_mean'] * efficiency_comparison['memory_used_mean'].clip(lower=1))
)

print("\n=== EFFICIENCY COMPARISON ===")
print(efficiency_comparison.sort_values('efficiency_score', ascending=False))

## 6. Analisi Qualitativa: Confronto Output

In [ ]:
# Analizziamo la qualità per categoria di task
quality_by_category = df.groupby(['category', 'model']).agg({
    'word_count': 'mean',
    'vocabulary_richness': 'mean',
    'avg_word_length': 'mean',
    'sentence_count': 'mean'
}).round(3)

print("=== QUALITY METRICS BY CATEGORY AND MODEL ===")
print(quality_by_category)

# Visualizziamo le differenze qualitative
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Ricchezza vocabolario per categoria
sns.barplot(data=df, x='category', y='vocabulary_richness', hue='model', ax=axes[0,0])
axes[0,0].set_title('Vocabulary Richness by Category')
axes[0,0].tick_params(axis='x', rotation=45)
axes[0,0].legend(title='Model')

# 2. Lunghezza media parole
sns.barplot(data=df, x='category', y='avg_word_length', hue='model', ax=axes[0,1])
axes[0,1].set_title('Average Word Length by Category')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].legend(title='Model')

# 3. Heatmap correlazioni
correlation_matrix = df[['generation_time', 'tokens_per_second', 'word_count', 'vocabulary_richness']].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[1,0])
axes[1,0].set_title('Correlation Matrix')

# 4. Performance vs Quality scatter
axes[1,1].scatter(df['tokens_per_second'], df['vocabulary_richness'], 
                  c=df['model'].astype('category').cat.codes, alpha=0.6)
axes[1,1].set_xlabel('Tokens per Second')
axes[1,1].set_ylabel('Vocabulary Richness')
axes[1,1].set_title('Performance vs Quality Trade-off')

# Aggiungi legenda per scatter
for i, model in enumerate(df['model'].unique()):
    axes[1,1].scatter([], [], c=f'C{i}', label=model)
axes[1,1].legend()

plt.tight_layout()
plt.show()

## 7. Esempi di Output Side-by-Side

In [ ]:
# Scegli un prompt per confronto dettagliato
example_category = "Creative Writing"
example_outputs = df[df['category'] == example_category]

print(f"=== SIDE-BY-SIDE COMPARISON: {example_category} ===")
print(f"Prompt: {test_prompts[0]['prompt']}\n")

for _, row in example_outputs.iterrows():
    print(f"**{row['model']} Model:**")
    print(f"Generated: {row['generated_text']}")
    print(f"Time: {row['generation_time']:.2f}s | Tokens/s: {row['tokens_per_second']:.1f} | Words: {row['word_count']}")
    print(f"Vocabulary richness: {row['vocabulary_richness']:.3f}\n")
    print("-" * 80 + "\n")

# Confronto tecnico
print("=== TECHNICAL COMPARISON ===")
technical_category = "Code Generation"
tech_outputs = df[df['category'] == technical_category]

for _, row in tech_outputs.iterrows():
    print(f"**{row['model']} - Code Generation:**")
    print(f"{row['generated_text']}")
    print(f"Performance: {row['tokens_per_second']:.1f} tok/s\n")


## 8. Trade-off Analysis e Raccomandazioni

In [ ]:
# Calcola score complessivo per ogni modello
def calculate_overall_score(row):
    # Normalizza metriche (0-1 scale)
    speed_score = (row['tokens_per_second_mean'] - df.groupby('model')['tokens_per_second'].mean().min()) / \
                  (df.groupby('model')['tokens_per_second'].mean().max() - df.groupby('model')['tokens_per_second'].mean().min())
    
    quality_score = (row['vocabulary_richness_mean'] - model_stats['vocabulary_richness_mean'].min()) / \
                    (model_stats['vocabulary_richness_mean'].max() - model_stats['vocabulary_richness_mean'].min())
    
    # Efficienza memoria (inverso)
    memory_score = 1 - ((row['memory_used_mean'] - model_stats['memory_used_mean'].min()) / \
                       (model_stats['memory_used_mean'].max() - model_stats['memory_used_mean'].min()))
    
    # Score combinato (peso equal)
    return (speed_score + quality_score + memory_score) / 3

model_rankings = model_stats.copy()
model_rankings['overall_score'] = model_rankings.apply(calculate_overall_score, axis=1)
model_rankings = model_rankings.sort_values('overall_score', ascending=False)

print("=== OVERALL MODEL RANKING ===")
print(model_rankings[['generation_time_mean', 'tokens_per_second_mean', 'vocabulary_richness_mean', 'overall_score']].round(3))

# Raccomandazioni per use case
recommendations = []

for model in models.keys():
    model_data = model_stats.loc[model]
    
    use_cases = []
    
    if model_data['tokens_per_second_mean'] > model_stats['tokens_per_second_mean'].median():
        use_cases.append("High-throughput applications")
    
    if model_data['memory_used_mean'] < model_stats['memory_used_mean'].median():
        use_cases.append("Resource-constrained environments")
        
    if model_data['vocabulary_richness_mean'] > model_stats['vocabulary_richness_mean'].median():
        use_cases.append("Creative/diverse content generation")
        
    if model_data['generation_time_mean'] < model_stats['generation_time_mean'].median():
        use_cases.append("Real-time applications")
    
    recommendations.append({
        'model': model,
        'best_for': ', '.join(use_cases) if use_cases else 'Baseline comparison',
        'overall_score': model_rankings.loc[model, 'overall_score']
    })

rec_df = pd.DataFrame(recommendations).sort_values('overall_score', ascending=False)

print("\n=== USE CASE RECOMMENDATIONS ===")
for _, row in rec_df.iterrows():
    print(f"**{row['model']}** (Score: {row['overall_score']:.3f})")
    print(f"  Best for: {row['best_for']}\n")

# Summary finale
print("=== KEY INSIGHTS ===")
fastest_model = model_stats['tokens_per_second_mean'].idxmax()
most_efficient = model_stats['memory_used_mean'].idxmin()
highest_quality = model_stats['vocabulary_richness_mean'].idxmax()

print(f"🚀 Fastest generation: {fastest_model} ({model_stats.loc[fastest_model, 'tokens_per_second_mean']:.1f} tok/s)")
print(f"💾 Most memory efficient: {most_efficient} ({model_stats.loc[most_efficient, 'memory_used_mean']:.1f} MB)")
print(f"✨ Highest vocabulary richness: {highest_quality} ({model_stats.loc[highest_quality, 'vocabulary_richness_mean']:.3f})")
print(f"🏆 Best overall: {rec_df.iloc[0]['model']} (balanced performance)")

## 9. Visualizzazione Finale: Dashboard Comparativo

In [ ]:
# Dashboard finale con tutte le metriche chiave
fig = plt.figure(figsize=(20, 12))

# Griglia 3x3
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Radar chart per ogni modello
ax1 = fig.add_subplot(gs[0, 0], projection='polar')
categories = ['Speed', 'Memory Efficiency', 'Quality', 'Consistency']
angles = np.linspace(0, 2*np.pi, len(categories), endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

for model in models.keys():
    values = [
        (model_stats.loc[model, 'tokens_per_second_mean'] - model_stats['tokens_per_second_mean'].min()) / 
        (model_stats['tokens_per_second_mean'].max() - model_stats['tokens_per_second_mean'].min()),
        1 - (model_stats.loc[model, 'memory_used_mean'] - model_stats['memory_used_mean'].min()) / 
        (model_stats['memory_used_mean'].max() - model_stats['memory_used_mean'].min()),
        (model_stats.loc[model, 'vocabulary_richness_mean'] - model_stats['vocabulary_richness_mean'].min()) / 
        (model_stats['vocabulary_richness_mean'].max() - model_stats['vocabulary_richness_mean'].min()),
        1 - (model_stats.loc[model, 'generation_time_std'] - model_stats['generation_time_std'].min()) / 
        (model_stats['generation_time_std'].max() - model_stats['generation_time_std'].min())
    ]
    values += values[:1]  # Complete the circle
    
    ax1.plot(angles, values, label=model, linewidth=2)
    ax1.fill(angles, values, alpha=0.25)

ax1.set_xticks(angles[:-1])
ax1.set_xticklabels(categories)
ax1.set_title('Model Comparison Radar', y=1.08)
ax1.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))

# 2. Speed comparison bar
ax2 = fig.add_subplot(gs[0, 1])
speed_data = model_stats['tokens_per_second_mean'].sort_values(ascending=False)
bars = ax2.bar(speed_data.index, speed_data.values, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
ax2.set_title('Generation Speed')
ax2.set_ylabel('Tokens/Second')
for bar, value in zip(bars, speed_data.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f'{value:.1f}', 
             ha='center', va='bottom')

# 3. Memory usage
ax3 = fig.add_subplot(gs[0, 2])
memory_data = model_stats['memory_used_mean'].sort_values()
bars = ax3.bar(memory_data.index, memory_data.values, color=['#d62728', '#9467bd', '#8c564b'])
ax3.set_title('Memory Usage')
ax3.set_ylabel('MB')
for bar, value in zip(bars, memory_data.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{value:.1f}', 
             ha='center', va='bottom')

# 4. Quality by task heatmap
ax4 = fig.add_subplot(gs[1, :])
quality_pivot = df.pivot_table(values='vocabulary_richness', index='category', columns='model')
sns.heatmap(quality_pivot, annot=True, fmt='.3f', cmap='viridis', ax=ax4)
ax4.set_title('Quality (Vocabulary Richness) by Task Category')

# 5. Performance distribution
ax5 = fig.add_subplot(gs[2, 0])
df.boxplot(column='generation_time', by='model', ax=ax5)
ax5.set_title('Generation Time Distribution')
ax5.set_xlabel('Model')
ax5.set_ylabel('Time (seconds)')

# 6. Quality distribution
ax6 = fig.add_subplot(gs[2, 1])
df.boxplot(column='vocabulary_richness', by='model', ax=ax6)
ax6.set_title('Quality Distribution')
ax6.set_xlabel('Model')
ax6.set_ylabel('Vocabulary Richness')

# 7. Overall score
ax7 = fig.add_subplot(gs[2, 2])
overall_scores = model_rankings['overall_score'].sort_values(ascending=False)
bars = ax7.bar(overall_scores.index, overall_scores.values, 
               color=['gold', 'silver', '#cd7f32'][:len(overall_scores)])
ax7.set_title('Overall Performance Score')
ax7.set_ylabel('Score')
for bar, value in zip(bars, overall_scores.values):
    ax7.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{value:.3f}', 
             ha='center', va='bottom')

plt.suptitle('Model Quantization Comparison Dashboard', fontsize=16, y=0.98)
plt.show()

## 10. Conclusioni e Best Practices

### Key Findings:

1. **Trade-off Fondamentale**: Esiste sempre un compromesso tra velocità, memoria e qualità
2. **Quantizzazione Efficace**: Modelli quantizzati possono mantenere buona qualità con minor consumo di risorse
3. **Task-Specific Performance**: Diversi tipi di quantizzazione funzionano meglio per task specifici

### Quando Usare Ogni Quantizzazione:

**FP32 (Full Precision)**:
- ✅ Massima qualità di generazione
- ✅ Stabilità numerica garantita
- ❌ Alto consumo memoria e tempo
- **Use case**: Ricerca, benchmark di riferimento

**FP16 (Half Precision)**:
- ✅ Buon compromesso qualità/velocità
- ✅ ~50% riduzione memoria
- ✅ Supportato da hardware moderno
- **Use case**: Produzione con GPU moderne

**INT8 (8-bit)**:
- ✅ Drastica riduzione memoria (~75%)
- ✅ Deployment su edge devices
- ❌ Potenziale degradazione qualità
- **Use case**: Mobile, IoT, edge computing

### Best Practices:

1. **Testa sempre**: La degradazione qualità varia per dataset e task
2. **Misura metriche rilevanti**: Non solo velocità, ma anche qualità output
3. **Considera il hardware**: CPU vs GPU supportano quantizzazioni diverse
4. **Bilancia i trade-off**: Prioritizza in base ai requisiti dell'applicazione
5. **Monitora in produzione**: Performance può variare con input reali


## Esercizi Pratici

**Esercizio 1**: Aggiungi INT4 quantization e confronta con gli altri modelli.

**Esercizio 2**: Testa lo stesso confronto su un task specifico (es. solo summarization) con metriche specializzate (ROUGE, BLEU).

**Esercizio 3**: Implementa un sistema di scoring che bilanci automaticamente speed/quality in base ai requisiti utente.

**Esercizio 4**: Estendi il confronto a modelli di dimensioni diverse (GPT-2 vs DistilGPT-2 vs modelli più grandi).

**Esercizio 5**: Misura il consumo energetico durante la generazione per una valutazione di sostenibilità.

---

**Fine del Notebook** | Module 13.3 - Quantization Comparison  
Data Visualization and Text Mining | Università Cattolica del Sacro Cuore